# GEO 371T/391: Climate Data - Spring 2026
# Assignment 10
# Exploring Internal Variability

*Developed by Cameron Cummins: 3/28/2026*

*Updated by Geeta Persad: 3/30/2026*

**In this notebook, we will expand our comparison of models and observations to include the effects of internal variability, leverage a large ensemble dataset: the NOAA Geophysical Fluid Dynamics Laboratory's SPEAR Large Ensemble**

Read more about SPEAR here: https://www.gfdl.noaa.gov/spear/

Read more about the SPEAR large ensemble here: https://www.gfdl.noaa.gov/spear_large_ensembles/

The SPEAR large ensemble consists of 30 members simulating the period 1921-2100 using SPEAR's Medium Resolution configuration (0.5$\deg$ land and atmosphere model horizontal resolution, 1$\deg$ ocean horizontal resolution).

**You will have the following tasks in this assignment, some of which we will work on in lecture and some of which you will complete outside of class time in the Assignment 10 Template document on Canvas:**

- **Task 1:** Read the SPEAR documentation at the links above and write a documentation paragraph addressing uncertainty in the Assignment 10 Template.
- **Task 2:** Plot different ensemble members in this notebook and then, in the Assignment 10 Template, paste in and write a caption for Figure 1 that describes similarities and differences between the ensemble members of your choice.
- **Task 3:** In the Assignment 10 Template, paste in and write a caption for Figure 2 that interprets the ensemble range in terms of the importance of internal variability in different regions
- **Task 4:** in the Assignment 10 Template, paste in and write a caption for Figure 3 and a paragraph interpreting what the figure says about the strengths and weaknesses of the GFDL SPEAR large ensemble's simulation of the historical period.

## Step 1: Run the below code block

### This code block imports our packages and data and defines our mapping projection.

In [ ]:
%%time
import xarray as xr
import numpy as np
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib
from os import listdir
matplotlib.style.use('fast')


class WinkelTripel(ccrs._WarpedRectangularProjection):
	"""
	Winkel-Tripel projection implementation for Cartopy
	"""

	def __init__(self, central_longitude=0.0, central_latitude=0.0, globe=None):
		globe = globe or ccrs.Globe(semimajor_axis=ccrs.WGS84_SEMIMAJOR_AXIS)
		proj4_params = [('proj', 'wintri'),
						('lon_0', central_longitude),
						('lat_0', central_latitude)]

		super(WinkelTripel, self).__init__(proj4_params, central_longitude, globe=globe)

	@property
	def threshold(self):
		return 1e4

head_input_dir = "/scratch/07644/oxygen/GEO371T_Post_Processed/obs_cmip_comparison"

gfdl_quantiles = xr.open_dataset(f"{head_input_dir}/GFDL-SPEAR-MED_historical_pr_quantiles.nc")
noaa_cpc_quantiles = xr.open_dataset(f"{head_input_dir}/NOAA-CPC_historical_pr_quantiles.nc")
gfdl_trends = xr.open_dataset(f"{head_input_dir}/GFDL-SPEAR-MED_historical_pr_trends.nc")
noaa_cpc_trends = xr.open_dataset(f"{head_input_dir}/NOAA-CPC_historical_pr_trends.nc")

head_input_dir = "/scratch/07644/oxygen/LANL_cleaned_data/metrics"

gfdl_hist_le = xr.open_zarr(f"{head_input_dir}/GFDL-SPEAR-MED_historical_pr-metrics.zarr")
gfdl_ssp585_le = xr.open_zarr(f"{head_input_dir}/GFDL-SPEAR-MED_ssp585_pr-metrics.zarr")
hist_yrly = gfdl_hist_le.weighted(np.cos(np.deg2rad(gfdl_hist_le.lat))).mean(dim=["lat", "lon"]).resample(time="YE").mean().compute()
future_yrly = gfdl_ssp585_le.weighted(np.cos(np.deg2rad(gfdl_hist_le.lat))).mean(dim=["lat", "lon"]).resample(time="YE").mean().compute()
far_future_avgs = gfdl_ssp585_le.sel(time=slice("2085-01-01", "2099-12-31")).mean(dim="time").sortby("member").compute()
hist_avgs = gfdl_hist_le.sel(time=slice("1985-01-01", "2014-12-31")).mean(dim="time").sortby("member").compute()

## Task 2: Exploring different ensemble members

The below code block creates (left) a plot of the globally averaged time series of precipitation from the GFDL SPEAR large ensemble, including all 30 members, and (right) a comparison of two individual ensemble members.

The ensemble members are identified in the code where you see `.sel(member="r1i1p1f1")` and `.sel(member="r2i1p1f1")`. The available 30 members can be accessed by replacing the number after the `r` with any integer between 1 and 30, inclusive, e.g.:
- `r1i1p1f1`
- `r2i1p1f1`
- `r3i1p1f1`
- ...
- `r29i1p1f1`
- `r30i1p1f1`

The code is currently set to look at ensemble members `r1` and `r2`. 

### STEPS
1. Play around with replacing the two ensemble members with others from the above list and replotting the figure until you find a combination that you think is interesting.
2. When you're happy with your plot save out Figure 1 by right clicking on the image.
3. In the Assignment 10 Template, paste in your Figure 1 and write a caption following the best practices established in prior assignments. Then write a short paragraph describing the similarities and differences between the two ensemble members you chose.


In [ ]:
%matplotlib inline
metric_var = "one_day_pr"

f, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), facecolor='w')

adj_hist_yrly = xr.concat([hist_yrly, future_yrly.sel(time=future_yrly.time[0])], dim="time", join='outer')
adj_hist_yrly[metric_var].mean(dim="member").plot(ax=ax1, color="black", label="Historical")
future_yrly[metric_var].mean(dim="member").plot(ax=ax1, color="#8036a7", label="SSP5 8.5")

for member in hist_yrly.member.values:
    adj_hist_yrly[metric_var].sel(member=member).plot(ax=ax1, color="black", linewidth=0.5, alpha=0.1)
    future_yrly[metric_var].sel(member=member).plot(ax=ax1, color="#8036a7", linewidth=0.5, alpha=0.1)

ax1.fill_between(adj_hist_yrly.time.values, adj_hist_yrly[metric_var].min(dim="member"), adj_hist_yrly[metric_var].max(dim="member"), alpha=0.1, color="black")
ax1.fill_between(future_yrly.time.values, future_yrly[metric_var].min(dim="member"), future_yrly[metric_var].max(dim="member"), alpha=0.1, color="#8036a7")

ax1.set_xlim(adj_hist_yrly.time.values[0], future_yrly.time.values[-1])
ax1.set_ylim(2.8, 3.2)
ax1.grid()
ax1.legend(fontsize=12)
ax1.set_xlabel("Time (Year)")
ax1.set_title("Full Ensemble Average and Range", fontsize=14)


adj_hist_yrly[metric_var].sel(member="r1i1p1f1").plot(ax=ax2, color="#003466", label="Realization/Member 1")
adj_hist_yrly[metric_var].sel(member="r2i1p1f1").plot(ax=ax2, color="#f69320", label="Realization/Member 2")

future_yrly[metric_var].sel(member="r1i1p1f1").plot(ax=ax2, color="#003466")
future_yrly[metric_var].sel(member="r2i1p1f1").plot(ax=ax2, color="#f69320")

ax2.set_xlim(adj_hist_yrly.time.values[0], future_yrly.time.values[-1])
ax2.set_ylim(2.8, 3.2)
ax2.grid()
ax2.legend(loc="upper left", fontsize=12)
ax2.set_xlabel("Time (Year)")
ax2.set_title("Comparing Two Members", fontsize=14)

f.suptitle("GFDL-SPEAR Large Ensemble Global Mean Precipitation", fontsize=20,y=1.025)

## Task 3: Mapping the ensemble range

The below plot maps the ensemble average (left) and ensemble range (right, difference between maximum values among ensemble members and minimum value among ensemble members) for the present-day (top, 1985-2014), future (middle, 2085-2099), and their difference (bottom).

For the ensemble ranges in the right-hand column, the top and middle panels are shown as the percent range (difference between max and min values among ensemble members relative to the ensemble average in each location) and the bottom panel is shown as the raw value rather than the percentage (i.e. the difference between the ensemble range in the future and the ensemble range in the present-day given in units of mm/day).

### STEPS
1. Run the below code block and save out the resulting figure.
2. In the Assignment 10 template, paste in your figure and write a figure caption following best practices and an interpretation paragraph addressing a set of provided questions.

In [ ]:
%matplotlib inline
from cartopy.util import add_cyclic_point

metric_var = "one_day_pr"

cb_label = "Percent range"

proj = WinkelTripel()
transform = ccrs.PlateCarree()

pr_levels = np.arange(0, 11, 1)
pr_range_levels = np.arange(0, 100, 5)
pr_diff_levels = np.arange(-3, 3, 0.1)

f, ((ax1, ax2), (ax3, ax4), (ax5, ax6)) = plt.subplots(3, 2, figsize=(20, 10), facecolor='w', subplot_kw=dict(projection=proj))

hist_avgs[metric_var].mean(dim="member").plot.contourf(ax=ax1, transform=transform, cmap="BuPu", levels=pr_levels)
((hist_avgs[metric_var].max(dim="member") - hist_avgs[metric_var].min(dim="member"))/hist_avgs[metric_var].mean(dim="member")*100).rename(cb_label).plot.contourf(ax=ax2, transform=transform, cmap="GnBu", levels=pr_range_levels, cbar_kwargs={'label': 'Percent range'})

far_future_avgs[metric_var].mean(dim="member").plot.contourf(ax=ax3, transform=transform, cmap="BuPu", levels=pr_levels)
((far_future_avgs[metric_var].max(dim="member") - far_future_avgs[metric_var].min(dim="member"))/far_future_avgs[metric_var].mean(dim="member")*100).rename(cb_label).plot.contourf(ax=ax4, transform=transform, cmap="GnBu", levels=pr_range_levels, cbar_kwargs={'label': 'Percent range'})

(far_future_avgs[metric_var].mean(dim="member")-hist_avgs[metric_var].mean(dim="member")).plot.contourf(ax=ax5, transform=transform, cmap="PuOr", levels=pr_diff_levels)
((far_future_avgs[metric_var].max(dim="member") - far_future_avgs[metric_var].min(dim="member"))-(hist_avgs[metric_var].max(dim="member") - hist_avgs[metric_var].min(dim="member"))).plot.contourf(ax=ax6, transform=transform, cmap="PuOr", levels=pr_diff_levels)


ax1.coastlines()
ax2.coastlines()
ax3.coastlines()
ax4.coastlines()
ax5.coastlines()
ax6.coastlines()

fz = 14
ax1.set_title("1985-2014 Ensemble Average", fontsize=fz, pad=15.0)
ax2.set_title("1985-2014 Ensemble Range relative to average", fontsize=fz, pad=15.0)
ax3.set_title("2085-2099 Ensemble Average", fontsize=fz, pad=15.0)
ax4.set_title("2085-2099 Ensemble Range relative to average", fontsize=fz, pad=15.0)
ax5.set_title("2085-2099 vs 1985-2014 Ensemble Average", fontsize=fz, pad=15.0)
ax6.set_title("2085-2099 vs 1985-2014 Ensemble Range", fontsize=fz, pad=15.0)

f.suptitle("GFDL-SPEAR Global Precipitation")

## Task 4: Observational comparison in the presence of internal variability

Observations provide a single realization of the evolution of the climate system, similar to individual ensemble members from a climate model dataset. When comparing models and observations, we often have to be careful to assess whether model-observation differences reflect true biases or whether they capture differences in the internal variability trajectory that manifested in the model versus the observations.

The below code block replicates a figure shown by Dr. Touma in her guest lecture (Lecture 16).

The figure show where trends in 1 day precipitation from the NOAA CPC observational dataset fall in the distribution of trends derived from each of the 30 members of the GFDL SPEAR ensemble. This is quantified by showing the percentile that the value of the NOAA CPC trend would fall into, if one were to construct a normal distribution of the trends from 30 ensemble members in GFDL SPEAR. A NOAA CPC trend percentile near 0.5, means that the observed trend is near the median of the modeled trends. A NOAA CPC trend percentile near or at 1, means that the observed trend is at the very high end of the modeled trends or exceeds the modeled trends. A NOAA CPC trend percentile near or at 0, means that the observed trend is at the very low end of the modeled trends or is smaller than any of the modeled trends.

### STEPS
1. Run the below block of code and save out the resulting figure.
2. In the Assignment 10 template, paste in your figure and write a figure caption following best practices and an interpretation paragraph addressing a set of provided questions.


In [ ]:
from scipy.stats import norm
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib

metric_var =  "one_day_pr_polyfit_coefficients"

noaa_cpc_trends_percs = (noaa_cpc_trends[metric_var] >= gfdl_trends[metric_var]).mean(dim='member')

cb_label = "Percentile"

proj = WinkelTripel()
transform = ccrs.PlateCarree()

plot_kwargs = dict(
    cmap="RdYlBu",
    transform=transform,
    vmax=1,
    vmin=0,
)

f, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6), facecolor='w', subplot_kw=dict(projection=proj))

noaa_cpc_trends_percs.rename(cb_label).where(noaa_cpc_trends[metric_var] > 0).sel(degree=1).plot(ax=ax1, **plot_kwargs)
noaa_cpc_trends_percs.rename(cb_label).where(noaa_cpc_trends[metric_var] < 0).sel(degree=1).plot(ax=ax2, **plot_kwargs)

ax1.set_title("NOAA CPC Trend > 0")
ax2.set_title("NOAA CPC Trend < 0")

ax1.coastlines()
ax2.coastlines()

f.suptitle("NOAA CPC 1-Day Precip. Trend Percentiles (Relative to GFDL Large Ensemble)", fontsize=20)